# 05 · Airport archetypes

**RQ3.** Do U.S. airports fall into distinct delay-risk archetypes once size and
cause mix are considered together?

Clustering is reported with its selection evidence — a silhouette sweep, a Ward
cross-check and a subsample stability score — because a k-means solution that
changes when a third of the airports are removed describes the sample, not the
airports.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

In [ ]:
from team01 import cluster, features, viz

panel = clean.completed(io.load_flights())
airports = io.load_airports()

matrix = features.airport_feature_matrix(panel, airports, min_departures=2000)
print(f"{len(matrix)} airports x {matrix.shape[1]} columns")
matrix.head(3)

In [ ]:
sweep = cluster.sweep_k(matrix)
chosen = cluster.choose_k(sweep)
print(f"chosen k = {chosen}")
sweep

In [ ]:
labelled, _, _ = cluster.fit(matrix, chosen)
names = cluster.name_clusters(labelled)
labelled["archetype"] = labelled["cluster"].map(names)

print(f"Ward cophenetic correlation {cluster.ward_check(matrix)['cophenetic_correlation']}")
print(cluster.stability(matrix, chosen))

cluster.cluster_table(labelled)[["cluster", "archetype", "airports",
                                 "mean_arr_delay", "share_delayed_60",
                                 "largest_members"]]

In [ ]:
viz.fig_silhouette(sweep, chosen)
viz.fig_cluster_scatter(labelled, names)

## Visual audit — Cairo's five questions

### Figure 09 · Choosing k by silhouette score

| Cairo question | Answer |
|---|---|
| **Guidelines** | A line over candidate k values with a marked chosen point, plus a horizontal reference at the minimum acceptable group size. Two decisions — score and minimum size — are shown as two separate marks rather than merged into one score. |
| **Truthful** | The chosen k = 4 is marked, and the stability verdict (mean ARI 0.7562 over 10 draws, minimum 0.4747) is printed next to it. A silhouette peak alone would overstate how well-separated the groups are. |
| **Functional** | The reader can see why 4 rather than the raw silhouette maximum, because the constraint is drawn rather than described. |
| **Beautiful** | One line, one marker, one annotation. Nothing else on the panel, so the single decision is unambiguous. |
| **Insightful** | The gap between the silhouette optimum and the stability test is the real finding: separation in feature space is not the same as reproducibility across subsamples, and the notebook says so instead of quoting the better-looking number. |

### Figure 10 · Airport clusters in feature space

| Cairo question | Answer |
|---|---|
| **Guidelines** | A two-dimensional projection with cluster membership as the primary encoding, point size as departure volume, and direct labels for the largest members. No third colour scale, because cluster identity already uses colour. |
| **Truthful** | The axes are named as the two standardised features actually plotted, so the reader is not invited to read distance in the projection as dissimilarity in ten dimensions. Labels describe regions of feature space — "tail-heavy", "airspace-constrained" — not airport types. |
| **Functional** | A reader can find a named airport and see which group it sits in, which is the question the figure is asked in practice. |
| **Beautiful** | Four hues, consistent with the archetype table, generous margins for labels, and a size legend that does not sit on top of points. |
| **Insightful** | Hub scale does not separate the groups — ATL and ORD sit with mid-size airports in the airspace-constrained group — so volume alone is not what distinguishes airports; the cause mix is. |